In [ ]:
pip install transformer_lens torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.3 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=10d44361ebb4790f01047675df7a98caa1d52ec42510ec0b0af1f1e73dd97341
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
"""
milestone2_evidence_graph_migration.py
======================================
Milestone 2 Architecture Migration: Simple Graph -> Epistemic Evidence Graph.
Ensures no overclaims by storing isolated experiments inside EvidenceLadder
and evaluating edge status via strict validation rules.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Epistemic Evidence Graph Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    """ثبت دقیق یک مداخله تجربی بدون تحمیل تفسیر علمی."""
    intervention_type: str            # e.g., "target_value_intervention", "residual_pre_patch", "residual_pre_ablation"
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None
    necessity_drop: Optional[float] = None

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    """نردبان شواهد چندسطحی - اجازه می‌دهد فیلدهای آزمایش‌نشده None باقی بمانند."""
    # Level 1
    directional_influence: Optional[InterventionEvidence] = None

    # Level 2
    residual_path_influence: Optional[InterventionEvidence] = None
    q_path: Optional[InterventionEvidence] = None
    k_path: Optional[InterventionEvidence] = None
    v_path: Optional[InterventionEvidence] = None

    # Mediation
    mediation: Optional[InterventionEvidence] = None

    # Level 3
    residual_contribution_dependence: Optional[InterventionEvidence] = None
    direct_edge_necessity: Optional[InterventionEvidence] = None


VALID_STATUSES = {
    "CANDIDATE",
    "DIRECTIONAL_INFLUENCE",
    "RESIDUAL_PATH_INFLUENCE",
    "MECHANISTIC_PATH_INFLUENCE",
    "DIRECT_EDGE",
    "NECESSARY_DIRECT_EDGE",
}


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        """ارزیابی خودکار و قانون‌مند وضعیت یال بدون مداخله دستی ایجنت."""
        ev = self.evidence

        if ev.directional_influence is None or abs(ev.directional_influence.null_z or 0) < 3.0 or abs(ev.directional_influence.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        if ev.residual_path_influence is None:
            return "DIRECTIONAL_INFLUENCE"

        # بررسی مسیرهای تفکیک‌شده Q/K/V
        if ev.q_path is None and ev.k_path is None and ev.v_path is None:
            return "RESIDUAL_PATH_INFLUENCE"

        # رزرو برای فازهای بعدی Q/K/V Path Isolation
        return "MECHANISTIC_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EVIDENCE GRAPH) FOR REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Candidate Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Discovered Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence
            l1_str = f"Raw: {ev.directional_influence.raw_effect:+6.4f} (Z: {ev.directional_influence.null_z:+5.2f})" if ev.directional_influence else "N/A"

            res_str = f"Target Δz: {ev.residual_path_influence.target_delta:+6.4f} | Output Δ: {ev.residual_path_influence.output_delta:+6.4f}" if ev.residual_path_influence else "Target Δz: N/A | Output Δ: N/A"

            dep_str = f"Drop: {ev.residual_contribution_dependence.necessity_drop:+6.4f}" if ev.residual_contribution_dependence else "Drop: N/A"

            q_str = "✓" if ev.q_path else "untested"
            k_str = "✓" if ev.k_path else "untested"
            v_str = "✓" if ev.v_path else "unisolated"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | Status: [{edge.status}]")
            print(f"     ├── L1 Directional Influence:    {l1_str}")
            print(f"     ├── L2 Residual Path Influence:  {res_str}")
            print(f"     ├── Q/K/V Isolation Paths:       Q: [{q_str}] | K: [{k_str}] | V: [{v_str}]")
            print(f"     └── L3 Residual Dependence:      {dep_str}\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_level2_direct_residual_path(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float]:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    def patch_resid_pre_hook(act, hook):
        act[0, pos, :] = act[0, pos, :] + delta_contrib
        return act

    target_z_patched = [None]
    def capture_target_z_hook(act, hook):
        target_z_patched[0] = act[0, pos, h_tgt, :].clone()
        return act

    hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_resid_pre_hook),
        (f"blocks.{l_tgt}.attn.hook_z", capture_target_z_hook)
    ]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    z_corrupted = corrupted_cache[f"blocks.{l_tgt}.attn.hook_z"][0, pos, h_tgt, :]
    z_patched = target_z_patched[0]
    target_head_z_delta = float((z_patched - z_corrupted).norm().item())

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    patched_diff = float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())
    output_logit_delta = patched_diff - base_diff

    return target_head_z_delta, output_logit_delta


def evaluate_level3_edge_necessity_ablation(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    clean_logits: torch.Tensor,
    clean_toks: torch.Tensor,
    pos: int = -1
) -> float:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)

    def ablate_resid_pre_hook(act, hook):
        act[0, pos, :] = act[0, pos, :] - contrib_clean
        return act

    hooks = [(f"blocks.{l_tgt}.hook_resid_pre", ablate_resid_pre_hook)]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        ablated_logits = model.run_with_hooks(clean_toks, fwd_hooks=hooks)

    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).item())
    ablated_diff = float((ablated_logits[0, pos, correct_id] - ablated_logits[0, pos, incorrect_id]).item())

    return clean_diff - ablated_diff


# =============================================================================
# 3. Evidence Discovery Pipeline
# =============================================================================

class EvidenceGraphEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            np.random.seed(42)
            indices = np.random.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def build_evidence_graph(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8,
        raw_influence_floor: float = 0.0100,
        direct_z_delta_threshold: float = 0.1000
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [MILESTONE 2 - EVIDENCE GRAPH] Building Graph for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- 1. Level 1: Directional Value Influence ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )
                null_z = (avg_raw - null_mean) / null_std

                evidence_ladder.directional_influence = InterventionEvidence(
                    intervention_type="target_value_intervention",
                    source=src,
                    target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v",
                    raw_effect=avg_raw,
                    null_mean=null_mean,
                    null_std=null_std,
                    null_z=null_z,
                    n_null_samples=k_null_samples
                )

                # --- 2. Level 2: Residual Path Influence ---
                if abs(null_z) >= 3.0 and abs(avg_raw) >= raw_influence_floor:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        head_z_delta, logit_delta = evaluate_level2_direct_residual_path(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        z_deltas.append(head_z_delta)
                        out_deltas.append(logit_delta)

                    avg_z_delta = float(np.mean(z_deltas))
                    avg_out_delta = float(np.mean(out_deltas))

                    if avg_z_delta >= direct_z_delta_threshold:
                        evidence_ladder.residual_path_influence = InterventionEvidence(
                            intervention_type="residual_pre_patch",
                            source=src,
                            target=tgt,
                            intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                            target_delta=avg_z_delta,
                            output_delta=avg_out_delta
                        )

                        # --- 3. Level 3: Residual Contribution Dependence ---
                        nec_drops = []
                        for task, clean_toks, _, clean_cache, _, _, clean_logits in task_data:
                            drop = evaluate_level3_edge_necessity_ablation(
                                self.model, src, tgt, task, clean_cache, clean_logits, clean_toks
                            )
                            nec_drops.append(drop)

                        avg_nec_drop = float(np.mean(nec_drops))
                        evidence_ladder.residual_contribution_dependence = InterventionEvidence(
                            intervention_type="residual_pre_ablation",
                            source=src,
                            target=tgt,
                            intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                            necessity_drop=avg_nec_drop
                        )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = EvidenceGraphEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.build_evidence_graph(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8,
        raw_influence_floor=0.0100,
        direct_z_delta_threshold=0.1000
    )

    evidence_graph.print_circuit_summary()

/tmp/ipykernel_837/3676478485.py:234: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 2 - EVIDENCE GRAPH] Building Graph for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (EVIDENCE GRAPH) FOR REGIME 'R1_IOI'
Total Candidate Nodes Evaluated: 6
Total Discovered Edges: 14

  ★ Pair L7H9  ──► L8H6  | Status: [CANDIDATE]
     ├── L1 Directional Influence:    Raw: +0.0000 (Z: +0.38)
     ├── L2 Residual Path Influence:  Target Δz: N/A | Output Δ: N/A
     ├── Q/K/V Isolation Paths:       Q: [untested] | K: [untested] | V: [unisolated]
     └── L3 Residual Dependence:      Drop: N/A

  ★ Pair L7H9  ──► L8H10 | Status: [CANDIDATE]
     ├── L1 Directional Influence:    Raw: -0.0044 (Z: -1.33)
     ├── L2 Residual Path Influence:  Target Δz: N/A | Output Δ: N/A
     ├── Q/K/V Isolation Paths:       Q: [untested] | K: [untested] | V: [unisolated]
     └── L3 Residual Dependence:      Drop: N/A

  ★ Pair L7H9  ──► L9H9  | Status: [CANDIDATE]
     ├── L1

In [ ]:
"""
milestone2_qkv_path_isolation.py
================================
Milestone 2 - Q/K/V Path Isolation Engine.
Isolates individual Q, K, and V attention pathways for candidate edges.
Updates EvidenceLadder to specify exact mechanistic channels (Q-path, K-path, V-path).
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Epistemic Evidence Graph Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None
    necessity_drop: Optional[float] = None

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    directional_influence: Optional[InterventionEvidence] = None
    residual_path_influence: Optional[InterventionEvidence] = None

    q_path: Optional[InterventionEvidence] = None
    k_path: Optional[InterventionEvidence] = None
    v_path: Optional[InterventionEvidence] = None

    mediation: Optional[InterventionEvidence] = None
    residual_contribution_dependence: Optional[InterventionEvidence] = None
    direct_edge_necessity: Optional[InterventionEvidence] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        ev = self.evidence

        if ev.directional_influence is None or abs(ev.directional_influence.null_z or 0) < 3.0 or abs(ev.directional_influence.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        if ev.residual_path_influence is None:
            return "DIRECTIONAL_INFLUENCE"

        if ev.q_path is not None or ev.k_path is not None or ev.v_path is not None:
            return "MECHANISTIC_PATH_INFLUENCE"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (Q/K/V ISOLATION) FOR REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Candidate Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Discovered Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence
            l1_str = f"Raw: {ev.directional_influence.raw_effect:+6.4f} (Z: {ev.directional_influence.null_z:+5.2f})" if ev.directional_influence else "N/A"
            res_str = f"Target Δz: {ev.residual_path_influence.target_delta:+6.4f} | Output Δ: {ev.residual_path_influence.output_delta:+6.4f}" if ev.residual_path_influence else "N/A"
            dep_str = f"Drop: {ev.residual_contribution_dependence.necessity_drop:+6.4f}" if ev.residual_contribution_dependence else "N/A"

            q_str = f"Output Δ: {ev.q_path.output_delta:+6.4f}" if ev.q_path else "untested"
            k_str = f"Output Δ: {ev.k_path.output_delta:+6.4f}" if ev.k_path else "untested"
            v_str = f"Output Δ: {ev.v_path.output_delta:+6.4f}" if ev.v_path else "unisolated"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | Status: [{edge.status}]")
            print(f"     ├── L1 Directional Influence:    {l1_str}")
            print(f"     ├── L2 Residual Path Influence:  {res_str}")
            print(f"     ├── Q-Path Isolation:            [{q_str}]")
            print(f"     ├── K-Path Isolation:            [{k_str}]")
            print(f"     ├── V-Path Isolation:            [{v_str}]")
            print(f"     └── L3 Residual Dependence:      {dep_str}\n")
        print("==================================================\n")


# =============================================================================
# 2. Q/K/V Isolation Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_qkv_path_isolation(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    path_type: str, # "q", "k", or "v"
    pos: int = -1
) -> Tuple[float, float]:
    """
    مداخله اختصاصی روی یکی از کانال‌های Q یا K یا V هد Target.
    """
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    # انتخاب ماتریس وزن مربوطه
    if path_type == "q":
        w_proj = model.W_Q[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_q"
    elif path_type == "k":
        w_proj = model.W_K[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_k"
    elif path_type == "v":
        w_proj = model.W_V[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_v"
    else:
        raise ValueError(f"Invalid path_type: {path_type}")

    delta_proj = delta_contrib @ w_proj

    def patch_qkv_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_proj
        return act

    target_z_patched = [None]
    def capture_target_z_hook(act, hook):
        target_z_patched[0] = act[0, pos, h_tgt, :].clone()
        return act

    hooks = [
        (hook_site, patch_qkv_hook),
        (f"blocks.{l_tgt}.attn.hook_z", capture_target_z_hook)
    ]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    z_corrupted = corrupted_cache[f"blocks.{l_tgt}.attn.hook_z"][0, pos, h_tgt, :]
    z_patched = target_z_patched[0]
    target_head_z_delta = float((z_patched - z_corrupted).norm().item())

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    patched_diff = float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())
    output_logit_delta = patched_diff - base_diff

    return target_head_z_delta, output_logit_delta


# =============================================================================
# 3. Pipeline Execution with Q/K/V Isolation
# =============================================================================

class QKVPathEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            np.random.seed(42)
            indices = np.random.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8,
        raw_influence_floor: float = 0.0100,
        qkv_threshold: float = 0.0100
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [MILESTONE 2 - Q/K/V ENGINE] Isolating Q/K/V Channels for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- 1. Level 1: Directional Value Scan ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )
                null_z = (avg_raw - null_mean) / null_std

                evidence_ladder.directional_influence = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=null_z, n_null_samples=k_null_samples
                )

                # --- 2. Level 2: Residual Path & Q/K/V Channel Isolation ---
                if abs(null_z) >= 3.0 and abs(avg_raw) >= raw_influence_floor:

                    # 2.a Residual Pre Patching
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.residual_path_influence = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas))
                    )

                    # 2.b Q-Path Isolation
                    q_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="q")
                        q_out_deltas.append(out_d)
                    avg_q = float(np.mean(q_out_deltas))
                    if abs(avg_q) >= qkv_threshold:
                        evidence_ladder.q_path = InterventionEvidence(intervention_type="q_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_q", output_delta=avg_q)

                    # 2.c K-Path Isolation
                    k_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="k")
                        k_out_deltas.append(out_d)
                    avg_k = float(np.mean(k_out_deltas))
                    if abs(avg_k) >= qkv_threshold:
                        evidence_ladder.k_path = InterventionEvidence(intervention_type="k_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_k", output_delta=avg_k)

                    # 2.d V-Path Isolation
                    v_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="v")
                        v_out_deltas.append(out_d)
                    avg_v = float(np.mean(v_out_deltas))
                    if abs(avg_v) >= qkv_threshold:
                        evidence_ladder.v_path = InterventionEvidence(intervention_type="v_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_v", output_delta=avg_v)

                    # --- 3. Level 3: Residual Contribution Dependence ---
                    nec_drops = []
                    for task, clean_toks, _, clean_cache, _, _, clean_logits in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        def ablate_res(act, hook):
                            act[0, -1, :] -= contrib_clean
                            return act
                        with torch.no_grad():
                            ablated = self.model.run_with_hooks(clean_toks, fwd_hooks=[(f"blocks.{tgt[0]}.hook_resid_pre", ablate_res)])
                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        c_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).item())
                        a_diff = float((ablated[0, -1, correct_id] - ablated[0, -1, incorrect_id]).item())
                        nec_drops.append(c_diff - a_diff)

                    evidence_ladder.residual_contribution_dependence = InterventionEvidence(
                        intervention_type="residual_pre_ablation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre", necessity_drop=float(np.mean(nec_drops))
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = QKVPathEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8,
        raw_influence_floor=0.0100,
        qkv_threshold=0.0100
    )

    evidence_graph.print_circuit_summary()

/tmp/ipykernel_837/2022707860.py:200: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 2 - Q/K/V ENGINE] Isolating Q/K/V Channels for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (Q/K/V ISOLATION) FOR REGIME 'R1_IOI'
Total Candidate Nodes Evaluated: 6
Total Discovered Edges: 14

  ★ Pair L7H9  ──► L8H6  | Status: [CANDIDATE]
     ├── L1 Directional Influence:    Raw: +0.0000 (Z: +0.38)
     ├── L2 Residual Path Influence:  N/A
     ├── Q-Path Isolation:            [untested]
     ├── K-Path Isolation:            [untested]
     ├── V-Path Isolation:            [unisolated]
     └── L3 Residual Dependence:      N/A

  ★ Pair L7H9  ──► L8H10 | Status: [CANDIDATE]
     ├── L1 Directional Influence:    Raw: -0.0044 (Z: -1.33)
     ├── L2 Residual Path Influence:  N/A
     ├── Q-Path Isolation:            [untested]
     ├── K-Path Isolation:            [untested]
     ├── V-Path Isolation:            [unisolated]
     └── L3 Residual Dependence:

In [ ]:
"""
milestone2_epistemic_evidence_graph.py
======================================
Milestone 2 - Refined Epistemic Evidence Graph.
Implements strict status Machine based on actual evidence logic:
- Differentiates Q_PATH, V_PATH, and MULTI_PATH Mechanistic Influences.
- Keeps DIRECT_EDGE and NECESSARY_DIRECT_EDGE locked until Attention Mediation is proven.
- Renames L1 to Target-Value Intervention to avoid overclaims.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    """ثبت دقیق داده‌های یک مداخله تجربی بدون فرض ادعای نهایی."""
    intervention_type: str            # e.g., "target_value_intervention", "residual_pre_patch", "q_path_patch", etc.
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None
    necessity_drop: Optional[float] = None

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    """نردبان شواهد چندسطحی با تفکیک دقیق کانال‌های مکانستیک."""
    # L1: Target-Value Intervention
    target_value_intervention: Optional[InterventionEvidence] = None

    # L2: Residual Path
    residual_path_influence: Optional[InterventionEvidence] = None

    # Q/K/V Isolation Paths
    q_path: Optional[InterventionEvidence] = None
    k_path: Optional[InterventionEvidence] = None
    v_path: Optional[InterventionEvidence] = None

    # Mediation
    attention_mediation: Optional[InterventionEvidence] = None

    # L3: Residual Contribution Dependence
    residual_contribution_dependence: Optional[InterventionEvidence] = None

    # Strictly reserved for future Edge-Level Isolation
    direct_edge_necessity: Optional[InterventionEvidence] = None


VALID_STATUSES = {
    "CANDIDATE",
    "TARGET_VALUE_INFLUENCE",
    "RESIDUAL_PATH_INFLUENCE",
    "Q_PATH_INFLUENCE",
    "K_PATH_INFLUENCE",
    "V_PATH_INFLUENCE",
    "MULTI_PATH_MECHANISTIC_INFLUENCE",
    "DIRECT_EDGE",
    "NECESSARY_DIRECT_EDGE",
}


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        """ماشین وضعیت خودمختار مبتنی بر شواهد واقعی ثبت‌شده."""
        ev = self.evidence

        # بررسی شرط L1
        l1 = ev.target_value_intervention
        if l1 is None or abs(l1.null_z or 0) < 3.0 or abs(l1.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        # بررسی شرط L2
        if ev.residual_path_influence is None:
            return "TARGET_VALUE_INFLUENCE"

        # بررسی کانال‌های Q/K/V
        active_paths = []
        if ev.q_path is not None and abs(ev.q_path.output_delta or 0) >= 0.0100:
            active_paths.append("Q")
        if ev.k_path is not None and abs(ev.k_path.output_delta or 0) >= 0.0100:
            active_paths.append("K")
        if ev.v_path is not None and abs(ev.v_path.output_delta or 0) >= 0.0100:
            active_paths.append("V")

        if len(active_paths) > 1:
            return "MULTI_PATH_MECHANISTIC_INFLUENCE"
        elif "Q" in active_paths:
            return "Q_PATH_INFLUENCE"
        elif "K" in active_paths:
            return "K_PATH_INFLUENCE"
        elif "V" in active_paths:
            return "V_PATH_INFLUENCE"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE GRAPH) - REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence
            l1_str = f"Raw: {ev.target_value_intervention.raw_effect:+6.4f} (Z: {ev.target_value_intervention.null_z:+5.2f})" if ev.target_value_intervention else "N/A"
            res_str = f"Target Δz: {ev.residual_path_influence.target_delta:+6.4f} | Output Δ: {ev.residual_path_influence.output_delta:+6.4f}" if ev.residual_path_influence else "N/A"
            dep_str = f"Drop: {ev.residual_contribution_dependence.necessity_drop:+6.4f}" if ev.residual_contribution_dependence else "N/A"

            q_str = f"Output Δ: {ev.q_path.output_delta:+6.4f}" if ev.q_path else "untested"
            k_str = f"Output Δ: {ev.k_path.output_delta:+6.4f}" if ev.k_path else "untested"
            v_str = f"Output Δ: {ev.v_path.output_delta:+6.4f}" if ev.v_path else "untested"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | Status: [{edge.status}]")
            print(f"     ├── L1 Target-Value Intervention: {l1_str}")
            print(f"     ├── L2 Residual Path Influence:   {res_str}")
            print(f"     ├── Q-Path Isolation:             [{q_str}]")
            print(f"     ├── K-Path Isolation:             [{k_str}]")
            print(f"     ├── V-Path Isolation:             [{v_str}]")
            print(f"     └── L3 Residual Dependence:       {dep_str}\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_qkv_path_isolation(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    path_type: str,
    pos: int = -1
) -> Tuple[float, float]:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    if path_type == "q":
        w_proj = model.W_Q[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_q"
    elif path_type == "k":
        w_proj = model.W_K[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_k"
    elif path_type == "v":
        w_proj = model.W_V[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_v"
    else:
        raise ValueError(f"Invalid path_type: {path_type}")

    delta_proj = delta_contrib @ w_proj

    def patch_qkv_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_proj
        return act

    target_z_patched = [None]
    def capture_target_z_hook(act, hook):
        target_z_patched[0] = act[0, pos, h_tgt, :].clone()
        return act

    hooks = [
        (hook_site, patch_qkv_hook),
        (f"blocks.{l_tgt}.attn.hook_z", capture_target_z_hook)
    ]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    z_corrupted = corrupted_cache[f"blocks.{l_tgt}.attn.hook_z"][0, pos, h_tgt, :]
    z_patched = target_z_patched[0]
    target_head_z_delta = float((z_patched - z_corrupted).norm().item())

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    patched_diff = float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())
    output_logit_delta = patched_diff - base_diff

    return target_head_z_delta, output_logit_delta


# =============================================================================
# 3. Pipeline Execution Engine
# =============================================================================

class RefinedEvidenceGraphEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)  # نمونه‌گیر مستقل بدون reset تصادفی داخلی

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8,
        raw_influence_floor: float = 0.0100,
        qkv_threshold: float = 0.0100
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [EPISTEMIC ENGINE] Running Evidence Pipeline for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # 1. Level 1: Target-Value Intervention Scan
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )
                null_z = (avg_raw - null_mean) / null_std

                evidence_ladder.target_value_intervention = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=null_z, n_null_samples=k_null_samples
                )

                # 2. Level 2: Residual Path & Q/K/V Isolation
                if abs(null_z) >= 3.0 and abs(avg_raw) >= raw_influence_floor:

                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.residual_path_influence = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas))
                    )

                    # Q-Path Isolation
                    q_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="q")
                        q_out_deltas.append(out_d)
                    avg_q = float(np.mean(q_out_deltas))
                    if abs(avg_q) >= qkv_threshold:
                        evidence_ladder.q_path = InterventionEvidence(intervention_type="q_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_q", output_delta=avg_q)

                    # K-Path Isolation
                    k_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="k")
                        k_out_deltas.append(out_d)
                    avg_k = float(np.mean(k_out_deltas))
                    if abs(avg_k) >= qkv_threshold:
                        evidence_ladder.k_path = InterventionEvidence(intervention_type="k_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_k", output_delta=avg_k)

                    # V-Path Isolation
                    v_out_deltas = []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        _, out_d = evaluate_qkv_path_isolation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="v")
                        v_out_deltas.append(out_d)
                    avg_v = float(np.mean(v_out_deltas))
                    if abs(avg_v) >= qkv_threshold:
                        evidence_ladder.v_path = InterventionEvidence(intervention_type="v_path_patch", source=src, target=tgt, intervention_site=f"blocks.{tgt[0]}.attn.hook_v", output_delta=avg_v)

                    # 3. Level 3: Residual Contribution Dependence
                    nec_drops = []
                    for task, clean_toks, _, clean_cache, _, _, clean_logits in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        def ablate_res(act, hook):
                            act[0, -1, :] -= contrib_clean
                            return act
                        with torch.no_grad():
                            ablated = self.model.run_with_hooks(clean_toks, fwd_hooks=[(f"blocks.{tgt[0]}.hook_resid_pre", ablate_res)])
                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        c_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).item())
                        a_diff = float((ablated[0, -1, correct_id] - ablated[0, -1, incorrect_id]).item())
                        nec_drops.append(c_diff - a_diff)

                    evidence_ladder.residual_contribution_dependence = InterventionEvidence(
                        intervention_type="residual_pre_ablation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre", necessity_drop=float(np.mean(nec_drops))
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = RefinedEvidenceGraphEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8,
        raw_influence_floor=0.0100,
        qkv_threshold=0.0100
    )

    evidence_graph.print_circuit_summary()

/tmp/ipykernel_837/2556135715.py:241: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [EPISTEMIC ENGINE] Running Evidence Pipeline for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE GRAPH) - REGIME 'R1_IOI'
Total Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6  | Status: [CANDIDATE]
     ├── L1 Target-Value Intervention: Raw: +0.0000 (Z: +5.60)
     ├── L2 Residual Path Influence:   N/A
     ├── Q-Path Isolation:             [untested]
     ├── K-Path Isolation:             [untested]
     ├── V-Path Isolation:             [untested]
     └── L3 Residual Dependence:       N/A

  ★ Pair L7H9  ──► L8H10 | Status: [CANDIDATE]
     ├── L1 Target-Value Intervention: Raw: -0.0044 (Z: -11.54)
     ├── L2 Residual Path Influence:   N/A
     ├── Q-Path Isolation:             [untested]
     ├── K-Path Isolation:             [untested]
     ├── V-Path Isolation:             [untested]
     └── L3 Residual Dependence:       

In [ ]:
"""
milestone2_attention_mediation_engine.py
========================================
Milestone 2 - Attention Mediation Engine.
Fixes Q/K/V isolation by measuring actual Attention Matrix changes (delta_A).
Locks edge status to RESIDUAL_PATH_INFLUENCE until Attention Mediation is proven.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    """ثبت دقیق داده‌های یک مداخله تجربی بدون پیش‌فرض علمی."""
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None
    attention_pattern_delta: Optional[float] = None  # norm(delta_A)
    necessity_drop: Optional[float] = None

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    """نردبان شواهد چندسطحی - با قفل کامل روی ادعاهای اثبات‌نشده."""
    target_value_intervention: Optional[InterventionEvidence] = None
    residual_path_influence: Optional[InterventionEvidence] = None

    q_mediated: Optional[InterventionEvidence] = None
    k_mediated: Optional[InterventionEvidence] = None
    v_mediated: Optional[InterventionEvidence] = None

    residual_contribution_dependence: Optional[InterventionEvidence] = None
    direct_edge_necessity: Optional[InterventionEvidence] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        """ماشین وضعیت سخت‌گیرانه برای جلوگیری از Overclaim."""
        ev = self.evidence

        l1 = ev.target_value_intervention
        if l1 is None or abs(l1.null_z or 0) < 3.0 or abs(l1.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        if ev.residual_path_influence is None:
            return "TARGET_VALUE_INFLUENCE"

        # بررسی وجود واسطه‌گری واقعی در ماتریس توجه
        q_valid = ev.q_mediated is not None and (ev.q_mediated.attention_pattern_delta or 0) > 0.01
        k_valid = ev.k_mediated is not None and (ev.k_mediated.attention_pattern_delta or 0) > 0.01
        v_valid = ev.v_mediated is not None and abs(ev.v_mediated.output_delta or 0) > 0.0100

        if q_valid and v_valid:
            return "MULTI_PATH_ATTENTION_MEDIATED"
        elif q_valid:
            return "Q_ATTENTION_MEDIATED"
        elif k_valid:
            return "K_ATTENTION_MEDIATED"
        elif v_valid:
            return "V_CONTENT_MEDIATED"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (ATTENTION MEDIATION) - REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence
            l1_str = f"Raw: {ev.target_value_intervention.raw_effect:+6.4f} (Z: {ev.target_value_intervention.null_z:+5.2f})" if ev.target_value_intervention else "N/A"
            res_str = f"Target Δz: {ev.residual_path_influence.target_delta:+6.4f} | Output Δ: {ev.residual_path_influence.output_delta:+6.4f}" if ev.residual_path_influence else "N/A"
            dep_str = f"Drop: {ev.residual_contribution_dependence.necessity_drop:+6.4f}" if ev.residual_contribution_dependence else "N/A"

            q_str = f"ΔAttn: {ev.q_mediated.attention_pattern_delta:6.4f} | Output Δ: {ev.q_mediated.output_delta:+6.4f}" if ev.q_mediated else "untested"
            v_str = f"Output Δ: {ev.v_mediated.output_delta:+6.4f}" if ev.v_mediated else "untested"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | Status: [{edge.status}]")
            print(f"     ├── L1 Target-Value Intervention: {l1_str}")
            print(f"     ├── L2 Residual Path Influence:   {res_str}")
            print(f"     ├── Q-Attention Mediation:        [{q_str}]")
            print(f"     ├── V-Content Mediation:          [{v_str}]")
            print(f"     └── L3 Residual Dependence:       {dep_str}\n")
        print("==================================================\n")


# =============================================================================
# 2. Attention Mediation Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_attention_mediation(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    path_type: str, # "q", "k", "v"
    pos: int = -1
) -> Tuple[float, float]:
    """
    سنجش تغییرات واقعی در الگوی توجه (delta_A) و خروجی لوگاریتم.
    """
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    if path_type == "q":
        w_proj = model.W_Q[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_q"
    elif path_type == "k":
        w_proj = model.W_K[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_k"
    elif path_type == "v":
        w_proj = model.W_V[l_tgt, h_tgt]
        hook_site = f"blocks.{l_tgt}.attn.hook_v"
    else:
        raise ValueError(f"Invalid path_type: {path_type}")

    delta_proj = delta_contrib @ w_proj

    def patch_qkv_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_proj
        return act

    # ثبت ماتریس توجه جدید جهت محاسبه delta_A
    pattern_patched = [None]
    def capture_pattern_hook(act, hook):
        pattern_patched[0] = act[0, h_tgt, pos, :].clone()
        return act

    hooks = [
        (hook_site, patch_qkv_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", capture_pattern_hook)
    ]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    # محاسبه انحراف ماتریس توجه نسبت به حالت corrupted
    pattern_corrupted = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, pos, :]
    attn_delta_norm = float((pattern_patched[0] - pattern_corrupted).norm().item())

    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())
    patched_diff = float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())
    output_logit_delta = patched_diff - base_diff

    return attn_delta_norm, output_logit_delta


# =============================================================================
# 3. Pipeline Execution Engine
# =============================================================================

class AttentionMediationPipeline:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8,
        raw_influence_floor: float = 0.0100
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [ATTENTION MEDIATION PIPELINE] Executing for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- 1. Level 1: Target-Value Intervention ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )
                null_z = (avg_raw - null_mean) / null_std

                evidence_ladder.target_value_intervention = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=null_z, n_null_samples=k_null_samples
                )

                # --- 2. Level 2: Residual Transmission & Attention Mediation ---
                if abs(null_z) >= 3.0 and abs(avg_raw) >= raw_influence_floor:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.residual_path_influence = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas))
                    )

                    # Q-Attention Mediation
                    q_attn_deltas, q_out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        attn_d, out_d = evaluate_attention_mediation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="q")
                        q_attn_deltas.append(attn_d)
                        q_out_deltas.append(out_d)

                    evidence_ladder.q_mediated = InterventionEvidence(
                        intervention_type="q_attention_mediation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_q",
                        attention_pattern_delta=float(np.mean(q_attn_deltas)), output_delta=float(np.mean(q_out_deltas))
                    )

                    # V-Content Mediation
                    v_attn_deltas, v_out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        attn_d, out_d = evaluate_attention_mediation(self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks, path_type="v")
                        v_attn_deltas.append(attn_d)
                        v_out_deltas.append(out_d)

                    evidence_ladder.v_mediated = InterventionEvidence(
                        intervention_type="v_content_mediation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_v",
                        attention_pattern_delta=float(np.mean(v_attn_deltas)), output_delta=float(np.mean(v_out_deltas))
                    )

                    # --- 3. Level 3: Residual Contribution Dependence ---
                    nec_drops = []
                    for task, clean_toks, _, clean_cache, _, _, clean_logits in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        def ablate_res(act, hook):
                            act[0, -1, :] -= contrib_clean
                            return act
                        with torch.no_grad():
                            ablated = self.model.run_with_hooks(clean_toks, fwd_hooks=[(f"blocks.{tgt[0]}.hook_resid_pre", ablate_res)])
                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        c_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).item())
                        a_diff = float((ablated[0, -1, correct_id] - ablated[0, -1, incorrect_id]).item())
                        nec_drops.append(c_diff - a_diff)

                    evidence_ladder.residual_contribution_dependence = InterventionEvidence(
                        intervention_type="residual_pre_ablation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre", necessity_drop=float(np.mean(nec_drops))
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = AttentionMediationPipeline("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8,
        raw_influence_floor=0.0100
    )

    evidence_graph.print_circuit_summary()

/tmp/ipykernel_837/28187690.py:213: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [ATTENTION MEDIATION PIPELINE] Executing for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (ATTENTION MEDIATION) - REGIME 'R1_IOI'
Total Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6  | Status: [CANDIDATE]
     ├── L1 Target-Value Intervention: Raw: +0.0000 (Z: +5.60)
     ├── L2 Residual Path Influence:   N/A
     ├── Q-Attention Mediation:        [untested]
     ├── V-Content Mediation:          [untested]
     └── L3 Residual Dependence:       N/A

  ★ Pair L7H9  ──► L8H10 | Status: [CANDIDATE]
     ├── L1 Target-Value Intervention: Raw: -0.0044 (Z: -11.54)
     ├── L2 Residual Path Influence:   N/A
     ├── Q-Attention Mediation:        [untested]
     ├── V-Content Mediation:          [untested]
     └── L3 Residual Dependence:       N/A

  ★ Pair L7H9  ──► L9H9  | Status: [CANDIDATE]
     ├── L1 Target-Value Intervention: Raw: -0.0000 (Z: -

In [ ]:
"""
milestone2_2_causal_attention_mediation.py
===========================================
Milestone 2.2 - Causal Attention Mediation Engine.
Implements Causal Clamping:
- Natural Intervention: Source patch -> Target Residual -> Natural Q/K/V & Attention -> Output (Y_natural)
- Clamped Intervention: Source patch -> Target Residual -> Force Attention A = A_corrupted -> Output (Y_clamped)
- Mediated Causal Effect: E_mediated = Y_natural - Y_clamped
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Epistemic Evidence Schema (Milestone 2.2)
# =============================================================================

@dataclass
class InterventionEvidence:
    """ثبت دقیق شواهد تجربی بدون تحمیل تفسیر علمی زودهنگام."""
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None
    attention_pattern_delta: Optional[float] = None  # norm(delta_A)

    # Causal Clamping Metrics
    natural_effect: Optional[float] = None
    clamped_effect: Optional[float] = None
    causal_mediated_effect: Optional[float] = None

    necessity_drop: Optional[float] = None
    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    """نردبان شواهد معرفت‌شناختی با تفکیک دقیق Perturbation از Mediation."""
    target_value_influence: Optional[InterventionEvidence] = None
    residual_path_influence: Optional[InterventionEvidence] = None

    # Perturbations
    q_perturbation: Optional[InterventionEvidence] = None
    v_perturbation: Optional[InterventionEvidence] = None

    # True Causal Mediation via Clamping
    causal_attention_mediation: Optional[InterventionEvidence] = None

    residual_contribution_dependence: Optional[InterventionEvidence] = None
    direct_edge_necessity: Optional[InterventionEvidence] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        """ماشین وضعیت خودمختار و محتاطانه (Epistemic Status Machine)."""
        ev = self.evidence

        l1 = ev.target_value_influence
        if l1 is None or abs(l1.null_z or 0) < 3.0 or abs(l1.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        if ev.residual_path_influence is None:
            return "TARGET_VALUE_INFLUENCE"

        # ارتقا به ATTENTION_MEDIATED تنها در صورت موفقیت آزمون Clamping
        cam = ev.causal_attention_mediation
        if cam is not None and abs(cam.causal_mediated_effect or 0) >= 0.0500:
            return "ATTENTION_MEDIATED"

        # در غیر این صورت، گزارش به عنوان Perturbation
        if ev.q_perturbation is not None and (ev.q_perturbation.attention_pattern_delta or 0) >= 0.0100:
            return "RESIDUAL_PATH_INFLUENCE_WITH_Q_PERTURBATION"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EPISTEMIC MILESTONE 2.2) - REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence
            l1_str = f"Raw: {ev.target_value_influence.raw_effect:+6.4f} (Z: {ev.target_value_influence.null_z:+5.2f})" if ev.target_value_influence else "N/A"
            res_str = f"Target Δz: {ev.residual_path_influence.target_delta:+6.4f} | Output Δ: {ev.residual_path_influence.output_delta:+6.4f}" if ev.residual_path_influence else "N/A"

            q_per_str = f"ΔAttn: {ev.q_perturbation.attention_pattern_delta:6.4f} | Output Δ: {ev.q_perturbation.output_delta:+6.4f}" if ev.q_perturbation else "untested"

            cam = ev.causal_attention_mediation
            cam_str = f"Natural: {cam.natural_effect:+6.4f} | Clamped: {cam.clamped_effect:+6.4f} | Mediated Δ: {cam.causal_mediated_effect:+6.4f}" if cam else "untested"

            dep_str = f"Drop: {ev.residual_contribution_dependence.necessity_drop:+6.4f}" if ev.residual_contribution_dependence else "N/A"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | Status: [{edge.status}]")
            print(f"     ├── L1 Target-Value Effect:       {l1_str}")
            print(f"     ├── L2 Residual Transmission:     {res_str}")
            print(f"     ├── Q-Attention Perturbation:     [{q_per_str}]")
            print(f"     ├── Causal Attention Mediation:   [{cam_str}]")
            print(f"     └── L3 Residual Dependence:       {dep_str}\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations & Causal Clamping Engine
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_causal_attention_clamping(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float, float]:
    """
    اجرای آزمون Causal Clamping:
    ۱. Natural Effect: تزریق سهم Source در Residual Stream و اجازه‌ی محاسبه طبیعی توجه.
    ۲. Clamped Effect: تزریق سهم Source اما قفل کردن اجباری ماتریس توجه روی حالت Corrupted.
    ۳. Mediated Effect = Natural Effect - Clamped Effect
    """
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # --- ۱. Natural Patching in Residual Stream ---
    def patch_res_hook(act, hook):
        act[0, pos, :] += delta_contrib
        return act

    with torch.no_grad():
        natural_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook)])

    natural_diff = float((natural_logits[0, pos, correct_id] - natural_logits[0, pos, incorrect_id]).item())
    natural_effect = natural_diff - base_diff

    # --- ۲. Clamped Attention Patching ---
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone()

    def clamp_attention_hook(act, hook):
        # اجبار ماتریس توجه به حالت Corrupted
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    clamped_hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", clamp_attention_hook)
    ]

    with torch.no_grad():
        clamped_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=clamped_hooks)

    clamped_diff = float((clamped_logits[0, pos, correct_id] - clamped_logits[0, pos, incorrect_id]).item())
    clamped_effect = clamped_diff - base_diff

    # --- ۳. Causal Mediated Effect ---
    mediated_effect = natural_effect - clamped_effect

    return natural_effect, clamped_effect, mediated_effect


# =============================================================================
# 3. Pipeline Execution Engine
# =============================================================================

class Milestone2_2_Engine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8,
        raw_influence_floor: float = 0.0100
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [MILESTONE 2.2 ENGINE] Running Causal Clamping Pipeline for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # 1. Level 1: Target-Value Effect
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )
                null_z = (avg_raw - null_mean) / null_std

                evidence_ladder.target_value_influence = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=null_z, n_null_samples=k_null_samples
                )

                # 2. Level 2: Residual Transmission & Causal Attention Clamping
                if abs(null_z) >= 3.0 and abs(avg_raw) >= raw_influence_floor:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.residual_path_influence = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas))
                    )

                    # Causal Clamping Test
                    nat_effs, clamp_effs, med_effs = [], [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        nat_e, clamp_e, med_e = evaluate_causal_attention_clamping(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        nat_effs.append(nat_e)
                        clamp_effs.append(clamp_e)
                        med_effs.append(med_e)

                    evidence_ladder.causal_attention_mediation = InterventionEvidence(
                        intervention_type="causal_attention_clamping", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_pattern",
                        natural_effect=float(np.mean(nat_effs)),
                        clamped_effect=float(np.mean(clamp_effs)),
                        causal_mediated_effect=float(np.mean(med_effs))
                    )

                    # 3. Level 3: Residual Contribution Dependence
                    nec_drops = []
                    for task, clean_toks, _, clean_cache, _, _, clean_logits in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        def ablate_res(act, hook):
                            act[0, -1, :] -= contrib_clean
                            return act
                        with torch.no_grad():
                            ablated = self.model.run_with_hooks(clean_toks, fwd_hooks=[(f"blocks.{tgt[0]}.hook_resid_pre", ablate_res)])
                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        c_diff = float((clean_logits[0, -1, correct_id] - clean_logits[0, -1, incorrect_id]).item())
                        a_diff = float((ablated[0, -1, correct_id] - ablated[0, -1, incorrect_id]).item())
                        nec_drops.append(c_diff - a_diff)

                    evidence_ladder.residual_contribution_dependence = InterventionEvidence(
                        intervention_type="residual_pre_ablation", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre", necessity_drop=float(np.mean(nec_drops))
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = Milestone2_2_Engine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8,
        raw_influence_floor=0.0100
    )

    evidence_graph.print_circuit_summary()

/tmp/ipykernel_837/2412600173.py:217: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 2.2 ENGINE] Running Causal Clamping Pipeline for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (EPISTEMIC MILESTONE 2.2) - REGIME 'R1_IOI'
Total Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6  | Status: [CANDIDATE]
     ├── L1 Target-Value Effect:       Raw: +0.0000 (Z: +5.60)
     ├── L2 Residual Transmission:     N/A
     ├── Q-Attention Perturbation:     [untested]
     ├── Causal Attention Mediation:   [untested]
     └── L3 Residual Dependence:       N/A

  ★ Pair L7H9  ──► L8H10 | Status: [CANDIDATE]
     ├── L1 Target-Value Effect:       Raw: -0.0044 (Z: -11.54)
     ├── L2 Residual Transmission:     N/A
     ├── Q-Attention Perturbation:     [untested]
     ├── Causal Attention Mediation:   [untested]
     └── L3 Residual Dependence:       N/A

  ★ Pair L7H9  ──► L9H9  | Status: [CANDIDATE]
     ├── L1 Target-Value Effect:      

In [ ]:
"""
milestone2_2_epistemic_engine.py
================================
Milestone 2.2 - Epistemic Attention Intervention Engine.
Features:
- Rigorous Interventional Attention Clamping via TransformerLens hook_pattern.
- Empirical P-value & Variance Floor check to eliminate inflated Z-scores.
- Strict Epistemic Evidence Classification (capping at INTERVENTIONAL_ATTENTION_EFFECT).
- Explicit tracking of Mediation Ratio and Descriptive Direction (AMPLIFYING, SUPPRESSIVE, OPPOSING, NEUTRAL).
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer


# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    """ثبت داده‌های مداخله تجربی بدون فرض ادعای مکانیزمی زودهنگام."""
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None
    empirical_p: Optional[float] = None
    null_variance_status: str = "VALID"

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None

    # Interventional Attention Metrics
    natural_effect: Optional[float] = None
    clamped_effect: Optional[float] = None
    mediated_effect: Optional[float] = None
    mediation_ratio: Optional[float] = None
    direction_label: str = "NEUTRAL"

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    """نردبان شواهد چندسطحی - قفل‌گذاری سخت‌گیرانه روی ادعاهای اثبات‌نشده."""
    level1_target_value: Optional[InterventionEvidence] = None
    level2_residual_path: Optional[InterventionEvidence] = None
    level3_attention_effect: Optional[InterventionEvidence] = None

    # Level 4 & 5 - Unestablished in current Milestone
    level4_qkv_attribution: Optional[Dict] = None
    level5_mechanistic_edge: Optional[Dict] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        """ماشین وضعیت خودمختار و محتاطانه (Epistemic Status Machine)."""
        ev = self.evidence

        # بررسی Level 1
        l1 = ev.level1_target_value
        if l1 is None or abs(l1.raw_effect or 0) < 0.0100:
            return "CANDIDATE"

        # چک معتبر بودن آماری L1 (بر اساس Z معتبر یا Empirical P)
        l1_valid = (l1.empirical_p is not None and l1.empirical_p <= 0.05) or \
                   (l1.null_z is not None and abs(l1.null_z) >= 3.0)

        if not l1_valid:
            return "CANDIDATE"

        # بررسی Level 2
        if ev.level2_residual_path is None:
            return "TARGET_VALUE_INFLUENCE"

        # بررسی Level 3
        l3 = ev.level3_attention_effect
        if l3 is not None and abs(l3.mediated_effect or 0) >= 0.0100:
            return "INTERVENTIONAL_ATTENTION_EFFECT"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self, n_tasks: int):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME '{self.regime_id}'")
        print(f"   Note: Evidence evaluated on {n_tasks} task(s). No broad generalization claimed.")
        print("==================================================")
        print(f"Total Candidate Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence

            # Formatting Level 1
            l1 = ev.level1_target_value
            if l1:
                z_str = f"{l1.null_z:+5.2f}" if l1.null_z is not None else f"[{l1.null_variance_status}]"
                p_str = f"{l1.empirical_p:.3f}" if l1.empirical_p is not None else "N/A"
                l1_str = f"Raw: {l1.raw_effect:+6.4f} | Null Mean: {l1.null_mean:+6.4f} (Std: {l1.null_std:.6f}) | Z: {z_str} | Emp p: {p_str}"
            else:
                l1_str = "N/A"

            # Formatting Level 2
            l2 = ev.level2_residual_path
            l2_str = f"Target Δz: {l2.target_delta:+6.4f} | Output Δ: {l2.output_delta:+6.4f}" if l2 else "N/A"

            # Formatting Level 3
            l3 = ev.level3_attention_effect
            if l3:
                ratio_str = f"{l3.mediation_ratio:+6.2f}" if l3.mediation_ratio is not None else "N/A"
                l3_str = (f"Natural: {l3.natural_effect:+6.4f} | Clamped: {l3.clamped_effect:+6.4f} | "
                          f"Mediated Δ: {l3.mediated_effect:+6.4f} | Ratio: {ratio_str} | Direction: [{l3.direction_label}]")
            else:
                l3_str = "N/A"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2}")
            print(f"     Status: [{edge.status}]")
            print(f"     ├── Level 1 (Target Value Influence):     {l1_str}")
            print(f"     ├── Level 2 (Residual Path Influence):     {l2_str}")
            print(f"     ├── Level 3 (Interventional Attn Effect):  {l3_str}")
            print(f"     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]")
            print(f"     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_attention_clamping_intervention(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float, float, float, str]:
    """
    اجرای مداخله کنترلی واسطه‌گری توجه:
    ۱. Natural Effect: انتقال سهم Source به Residual ورودی Target و بازمحاسبه طبیعی Attention Pattern.
    ۲. Clamped Effect: همان انتقال سهم در Residual ورودی Target، ولی جایگزینی اجباری Attention Pattern با حالت Corrupted Baseline.
    """
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # --- ۱. Natural Intervention ---
    def patch_res_hook(act, hook):
        act[0, pos, :] += delta_contrib
        return act

    with torch.no_grad():
        natural_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook)])

    natural_diff = float((natural_logits[0, pos, correct_id] - natural_logits[0, pos, incorrect_id]).item())
    natural_effect = natural_diff - base_diff

    # --- ۲. Clamped Intervention ---
    # استخراج دقیقا همان الگوی Softmax شده توجه در حالت Corrupted Baseline
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone()

    def clamp_attention_pattern_hook(act, hook):
        # جایگزینی الگوی توجه Target Head با Baseline
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    clamped_hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", clamp_attention_pattern_hook)
    ]

    with torch.no_grad():
        clamped_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=clamped_hooks)

    clamped_diff = float((clamped_logits[0, pos, correct_id] - clamped_logits[0, pos, incorrect_id]).item())
    clamped_effect = clamped_diff - base_diff

    # --- ۳. محاسبه اثر واسطه‌ای و برچسب توصیفی ---
    mediated_effect = natural_effect - clamped_effect

    abs_natural = abs(natural_effect)
    mediation_ratio = (mediated_effect / abs_natural) if abs_natural > 1e-6 else 0.0

    # تعیین برچسب توصیفی جهت اثر
    if abs(mediated_effect) < 0.0100:
        direction_label = "NEUTRAL"
    elif np.sign(natural_effect) == np.sign(mediated_effect):
        direction_label = "AMPLIFYING" if abs(natural_effect) > abs(clamped_effect) else "SUPPRESSIVE"
    else:
        direction_label = "OPPOSING"

    return natural_effect, clamped_effect, mediated_effect, mediation_ratio, direction_label


# =============================================================================
# 3. Pipeline Execution Engine
# =============================================================================

class EpistemicEvidenceEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 64
    ) -> Tuple[float, float, Optional[float], str, List[float]]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        null_mean = float(np.mean(null_deltas))
        null_std = float(np.std(null_deltas))

        # کالیبراسیون واریانس جهت جلوگیری از Z-scoreهای منفجرشده
        if null_std < 1e-6:
            variance_status = "NULL_VARIANCE_TOO_SMALL"
            null_z = None
        else:
            variance_status = "VALID"
            null_z = None  # به‌جای مقدار خام، در بخش ارزیابی اصلی با کنترل محاسبه می‌شود

        return null_mean, null_std, null_z, variance_status, null_deltas

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 64,
        raw_influence_floor: float = 0.0100
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])
        n_tasks = len(tasks)

        print(f"\n⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime '{regime_id}' across {n_tasks} task(s)")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- Level 1: Target-Value Influence ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std, _, var_status, null_deltas = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )

                # محاسبه Empirical P-value
                if len(null_deltas) > 0:
                    empirical_p = float((np.sum(np.abs(null_deltas) >= abs(avg_raw)) + 1) / (len(null_deltas) + 1))
                else:
                    empirical_p = None

                # محاسبه Z معتبر تنها در صورت عدم وجود Degenerate Variance
                if var_status == "VALID":
                    computed_z = (avg_raw - null_mean) / null_std
                else:
                    computed_z = None

                evidence_ladder.level1_target_value = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=computed_z,
                    empirical_p=empirical_p, null_variance_status=var_status,
                    n_null_samples=k_null_samples, metadata={"n_tasks": n_tasks}
                )

                # --- Level 2: Residual Path Influence ---
                l1_valid = (empirical_p is not None and empirical_p <= 0.05) or (computed_z is not None and abs(computed_z) >= 3.0)

                if l1_valid and abs(avg_raw) >= raw_influence_floor:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.level2_residual_path = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas)),
                        metadata={"n_tasks": n_tasks}
                    )

                    # --- Level 3: Interventional Attention Effect (Clamping Test) ---
                    nat_effs, clamp_effs, med_effs, ratios, directions = [], [], [], [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        nat_e, clamp_e, med_e, ratio, dir_label = evaluate_attention_clamping_intervention(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        nat_effs.append(nat_e)
                        clamp_effs.append(clamp_e)
                        med_effs.append(med_e)
                        ratios.append(ratio)
                        directions.append(dir_label)

                    avg_med = float(np.mean(med_effs))
                    avg_nat = float(np.mean(nat_effs))
                    avg_ratio = (avg_med / abs(avg_nat)) if abs(avg_nat) > 1e-6 else 0.0

                    # برچسب توصیفی کلی بر اساس میانگین‌ها
                    if abs(avg_med) < 0.0100:
                        overall_dir = "NEUTRAL"
                    elif np.sign(avg_nat) == np.sign(avg_med):
                        overall_dir = "AMPLIFYING" if abs(avg_nat) > abs(float(np.mean(clamp_effs))) else "SUPPRESSIVE"
                    else:
                        overall_dir = "OPPOSING"

                    evidence_ladder.level3_attention_effect = InterventionEvidence(
                        intervention_type="interventional_attention_clamping", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_pattern",
                        natural_effect=avg_nat,
                        clamped_effect=float(np.mean(clamp_effs)),
                        mediated_effect=avg_med,
                        mediation_ratio=avg_ratio,
                        direction_label=overall_dir,
                        metadata={"n_tasks": n_tasks}
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 4. Pipeline Execution
# =============================================================================

if __name__ == "__main__":
    engine = EpistemicEvidenceEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=64,
        raw_influence_floor=0.0100
    )

    evidence_graph.print_circuit_summary(n_tasks=len(IOI_TASKS))

/tmp/ipykernel_837/4146516614.py:247: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime 'R1_IOI' across 2 task(s)
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME 'R1_IOI'
   Note: Evidence evaluated on 2 task(s). No broad generalization claimed.
Total Candidate Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6 
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: +0.0000 | Null Mean: -0.0000 (Std: 0.000016) | Z: +1.04 | Emp p: 0.077
     ├── Level 2 (Residual Path Influence):     N/A
     ├── Level 3 (Interventional Attn Effect):  N/A
     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]
     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]

  ★ Pair L7H9  ──► L8H10
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: -0.0044 | Null Mean: +0.0003 (Std: 0.001922) | Z: -2.44 | Emp p: 0.031
     ├── Level 2 (Residual

In [ ]:
# =============================================================================
# Global Configuration & Physical Thresholds
# =============================================================================
RAW_INFLUENCE_FLOOR: float = 0.0100      # حداقل اثر فیزیکی خام
MEDIATION_EFFECT_FLOOR: float = 0.0100   # حداقل اثر واسطه‌ای
STATISTICAL_P_THRESHOLD: float = 0.05    # آستانه معناداری آماری (Empirical P-value)
DEFAULT_K_NULL_SAMPLES: int = 64         # تعداد نمونه‌های توزیع نویز

In [ ]:
"""
milestone3_predictive_benchmark.py
==================================
Milestone 3 - Predictive Self-Model Benchmark.
1. Centralizes physical and statistical thresholds.
2. Uses SelfCircuitGraph evidence to PREDICT multi-node counterfactual interventions.
3. Evaluates Prediction Error (MAE, R^2) against ground-truth model outputs.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# ثابت‌های یکپارچه
RAW_INFLUENCE_FLOOR: float = 0.0100
MEDIATION_EFFECT_FLOOR: float = 0.0100
STATISTICAL_P_THRESHOLD: float = 0.05
DEFAULT_K_NULL_SAMPLES: int = 64


class SelfModelPredictor:
    """
    موتور پیش‌بینی خودشناس:
    پیش‌بینی اثر مداخلات ترکیبی ندیده صرفاً با استفاده از پرونده‌های شواهد در SelfCircuitGraph.
    """
    def __init__(self, circuit_graph):
        self.graph = circuit_graph

    def predict_joint_intervention(self, source_nodes: List[Tuple[int, int]], target_node: Tuple[int, int]) -> float:
        """
        برآورد اثر مداخله هم‌زمان چند هد بر اساس مجموع برداری اثرات واسطه‌گری ثبت‌شده.
        """
        predicted_effect = 0.0
        for src in source_nodes:
            # پیدا کردن یال مربوطه در گراف
            edge = next((e for e in self.graph.edges if e.source == src and e.target == target_node), None)
            if edge and edge.evidence.level3_attention_effect:
                # استفاده از Natural Effect ثبت‌شده در گراف
                predicted_effect += edge.evidence.level3_attention_effect.natural_effect
        return predicted_effect


def evaluate_predictive_benchmark(
    model: HookedTransformer,
    predictor: SelfModelPredictor,
    joint_sources: List[Tuple[int, int]],
    target: Tuple[int, int],
    tasks: List[Dict]
) -> Dict[str, float]:
    """
    ارزیابی دقت پیش‌بینی گراف علّی در برابر مداخله هم‌زمان واقعی روی مدل.
    """
    l_tgt, h_tgt = target
    actual_effects = []
    predicted_effects = []

    # ۱. محاسبه پیش‌بینی گراف علّی (بدون اجرای مداخله روی مدل)
    predicted_effect = predictor.predict_joint_intervention(joint_sources, target)

    # ۲. اجرای مداخله هم‌زمان واقعی روی مدل جهت سنجش Ground Truth
    for task in tasks:
        clean_toks = model.to_tokens(task["clean"])
        corrupted_toks = model.to_tokens(task["corrupted"])

        with torch.no_grad():
            _, clean_cache = model.run_with_cache(clean_toks)
            base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

        # محاسبه مجموع delta_contrib برای تمام هدهای منبع در مداخله ترکیبی
        total_delta_contrib = 0.0
        for src in joint_sources:
            l_src, h_src = src
            z_clean = clean_cache[f"blocks.{l_src}.attn.hook_z"][0, -1, h_src, :]
            z_corrupt = corrupted_cache[f"blocks.{l_src}.attn.hook_z"][0, -1, h_src, :]
            contrib_clean = z_clean @ model.W_O[l_src, h_src]
            contrib_corrupt = z_corrupt @ model.W_O[l_src, h_src]
            total_delta_contrib += (contrib_clean - contrib_corrupt)

        def patch_joint_res_hook(act, hook):
            act[0, -1, :] += total_delta_contrib
            return act

        with torch.no_grad():
            patched_logits = model.run_with_hooks(
                corrupted_toks,
                fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_joint_res_hook)]
            )

        correct_id = int(model.to_single_token(task["correct"]))
        incorrect_id = int(model.to_single_token(task["incorrect"]))
        base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
        patched_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

        actual_effects.append(patched_diff - base_diff)
        predicted_effects.append(predicted_effect)

    actual_mean = float(np.mean(actual_effects))
    pred_mean = float(np.mean(predicted_effects))
    mae = float(np.abs(actual_mean - pred_mean))

    return {
        "actual_joint_effect": actual_mean,
        "predicted_joint_effect": pred_mean,
        "mae": mae
    }

In [ ]:
"""
milestone3_predictive_benchmark_full.py
======================================
Milestone 3 - Predictive Self-Model Benchmark Engine.
1. Centralized physical and statistical thresholds.
2. Builds Epistemic Evidence Graph (Milestone 2.2 pipeline).
3. Uses SelfModelPredictor to predict multi-node joint interventions (L9H9 + L10H7 -> L11H10).
4. Evaluates Prediction Error (MAE) against ground-truth model outputs.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# Global Configuration & Physical Thresholds
# =============================================================================
RAW_INFLUENCE_FLOOR: float = 0.0100      # حداقل اثر فیزیکی خام
MEDIATION_EFFECT_FLOOR: float = 0.0100   # حداقل اثر واسطه‌ای
STATISTICAL_P_THRESHOLD: float = 0.05    # آستانه معناداری آماری (Empirical P-value)
DEFAULT_K_NULL_SAMPLES: int = 64         # تعداد نمونه‌های توزیع نویز


# =============================================================================
# 1. Epistemic Evidence Schema
# =============================================================================

@dataclass
class InterventionEvidence:
    intervention_type: str
    source: Tuple[int, int]
    target: Tuple[int, int]
    intervention_site: str

    raw_effect: Optional[float] = None
    null_mean: Optional[float] = None
    null_std: Optional[float] = None
    null_z: Optional[float] = None
    empirical_p: Optional[float] = None
    null_variance_status: str = "VALID"

    target_delta: Optional[float] = None
    output_delta: Optional[float] = None

    natural_effect: Optional[float] = None
    clamped_effect: Optional[float] = None
    mediated_effect: Optional[float] = None
    mediation_ratio: Optional[float] = None
    direction_label: str = "NEUTRAL"

    n_null_samples: int = 0
    metadata: Dict = field(default_factory=dict)


@dataclass
class EvidenceLadder:
    level1_target_value: Optional[InterventionEvidence] = None
    level2_residual_path: Optional[InterventionEvidence] = None
    level3_attention_effect: Optional[InterventionEvidence] = None

    level4_qkv_attribution: Optional[Dict] = None
    level5_mechanistic_edge: Optional[Dict] = None


@dataclass
class EvidenceEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceLadder = field(default_factory=EvidenceLadder)
    metadata: Dict = field(default_factory=dict)

    @property
    def status(self) -> str:
        ev = self.evidence

        l1 = ev.level1_target_value
        if l1 is None or abs(l1.raw_effect or 0) < RAW_INFLUENCE_FLOOR:
            return "CANDIDATE"

        l1_valid = (l1.empirical_p is not None and l1.empirical_p <= STATISTICAL_P_THRESHOLD) or \
                   (l1.null_z is not None and abs(l1.null_z) >= 3.0)

        if not l1_valid:
            return "CANDIDATE"

        if ev.level2_residual_path is None:
            return "TARGET_VALUE_INFLUENCE"

        l3 = ev.level3_attention_effect
        if l3 is not None and abs(l3.mediated_effect or 0) >= MEDIATION_EFFECT_FLOOR:
            return "INTERVENTIONAL_ATTENTION_EFFECT"

        return "RESIDUAL_PATH_INFLUENCE"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[EvidenceEdge] = field(default_factory=list)

    def add_edge(self, edge: EvidenceEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self, n_tasks: int):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME '{self.regime_id}'")
        print(f"   Note: Evidence evaluated on {n_tasks} task(s). No broad generalization claimed.")
        print("==================================================")
        print(f"Total Candidate Nodes Evaluated: {len(self.nodes)}")
        print(f"Total Tracked Edges: {len(self.edges)}\n")

        for edge in self.edges:
            ev = edge.evidence

            l1 = ev.level1_target_value
            if l1:
                z_str = f"{l1.null_z:+5.2f}" if l1.null_z is not None else f"[{l1.null_variance_status}]"
                p_str = f"{l1.empirical_p:.3f}" if l1.empirical_p is not None else "N/A"
                l1_str = f"Raw: {l1.raw_effect:+6.4f} | Null Mean: {l1.null_mean:+6.4f} (Std: {l1.null_std:.6f}) | Z: {z_str} | Emp p: {p_str}"
            else:
                l1_str = "N/A"

            l2 = ev.level2_residual_path
            l2_str = f"Target Δz: {l2.target_delta:+6.4f} | Output Δ: {l2.output_delta:+6.4f}" if l2 else "N/A"

            l3 = ev.level3_attention_effect
            if l3:
                ratio_str = f"{l3.mediation_ratio:+6.2f}" if l3.mediation_ratio is not None else "N/A"
                l3_str = (f"Natural: {l3.natural_effect:+6.4f} | Clamped: {l3.clamped_effect:+6.4f} | "
                          f"Mediated Δ: {l3.mediated_effect:+6.4f} | Ratio: {ratio_str} | Direction: [{l3.direction_label}]")
            else:
                l3_str = "N/A"

            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2}")
            print(f"     Status: [{edge.status}]")
            print(f"     ├── Level 1 (Target Value Influence):     {l1_str}")
            print(f"     ├── Level 2 (Residual Path Influence):     {l2_str}")
            print(f"     ├── Level 3 (Interventional Attn Effect):  {l3_str}")
            print(f"     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]")
            print(f"     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]\n")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations & Pipeline Engine
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_attention_clamping_intervention(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    base_logits: torch.Tensor,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> Tuple[float, float, float, float, str]:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))
    base_diff = float((base_logits[0, pos, correct_id] - base_logits[0, pos, incorrect_id]).item())

    # 1. Natural
    def patch_res_hook(act, hook):
        act[0, pos, :] += delta_contrib
        return act

    with torch.no_grad():
        natural_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook)])

    natural_diff = float((natural_logits[0, pos, correct_id] - natural_logits[0, pos, incorrect_id]).item())
    natural_effect = natural_diff - base_diff

    # 2. Clamped
    corrupted_pattern = corrupted_cache[f"blocks.{l_tgt}.attn.hook_pattern"][0, h_tgt, :, :].clone()

    def clamp_attention_pattern_hook(act, hook):
        act[0, h_tgt, :, :] = corrupted_pattern
        return act

    clamped_hooks = [
        (f"blocks.{l_tgt}.hook_resid_pre", patch_res_hook),
        (f"blocks.{l_tgt}.attn.hook_pattern", clamp_attention_pattern_hook)
    ]

    with torch.no_grad():
        clamped_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=clamped_hooks)

    clamped_diff = float((clamped_logits[0, pos, correct_id] - clamped_logits[0, pos, incorrect_id]).item())
    clamped_effect = clamped_diff - base_diff

    mediated_effect = natural_effect - clamped_effect
    abs_natural = abs(natural_effect)
    mediation_ratio = (mediated_effect / abs_natural) if abs_natural > 1e-6 else 0.0

    if abs(mediated_effect) < MEDIATION_EFFECT_FLOOR:
        direction_label = "NEUTRAL"
    elif np.sign(natural_effect) == np.sign(mediated_effect):
        direction_label = "AMPLIFYING" if abs(natural_effect) > abs(clamped_effect) else "SUPPRESSIVE"
    else:
        direction_label = "OPPOSING"

    return natural_effect, clamped_effect, mediated_effect, mediation_ratio, direction_label


class EpistemicEvidenceEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()
        self.rng = np.random.default_rng(42)

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> Tuple[float, float, Optional[float], str, List[float]]:
        l_tgt, _ = target
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            indices = self.rng.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[target[0], target[1]]

                def patch_v(act, hook):
                    act[0, -1, target[1], :] += delta_v
                    return act

                with torch.no_grad():
                    patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{target[0]}.attn.hook_v", patch_v)])

                p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                task_effects.append(p_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        null_mean = float(np.mean(null_deltas))
        null_std = float(np.std(null_deltas))

        if null_std < 1e-6:
            variance_status = "NULL_VARIANCE_TOO_SMALL"
            null_z = None
        else:
            variance_status = "VALID"
            null_z = None

        return null_mean, null_std, null_z, variance_status, null_deltas

    def execute_pipeline(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = DEFAULT_K_NULL_SAMPLES
    ) -> SelfCircuitGraph:

        graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])
        n_tasks = len(tasks)

        print(f"\n⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime '{regime_id}' across {n_tasks} task(s)")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits, clean_logits))

        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                evidence_ladder = EvidenceLadder()

                # --- Level 1 ---
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                    contrib_corrupted = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                    delta_v = (contrib_clean - contrib_corrupted) @ self.model.W_V[tgt[0], tgt[1]]

                    def patch_v(act, hook):
                        act[0, -1, tgt[1], :] += delta_v
                        return act

                    with torch.no_grad():
                        patched = self.model.run_with_hooks(corrupted_toks, fwd_hooks=[(f"blocks.{tgt[0]}.attn.hook_v", patch_v)])
                    p_diff = float((patched[0, -1, correct_id] - patched[0, -1, incorrect_id]).item())
                    raw_effects.append(p_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))
                null_mean, null_std, _, var_status, null_deltas = self.generate_clean_null_distribution(
                    target=tgt, task_data=task_data, exclude_sources=target_candidates, k_samples=k_null_samples
                )

                if len(null_deltas) > 0:
                    empirical_p = float((np.sum(np.abs(null_deltas) >= abs(avg_raw)) + 1) / (len(null_deltas) + 1))
                else:
                    empirical_p = None

                computed_z = (avg_raw - null_mean) / null_std if var_status == "VALID" else None

                evidence_ladder.level1_target_value = InterventionEvidence(
                    intervention_type="target_value_intervention", source=src, target=tgt,
                    intervention_site=f"blocks.{tgt[0]}.attn.hook_v", raw_effect=avg_raw,
                    null_mean=null_mean, null_std=null_std, null_z=computed_z,
                    empirical_p=empirical_p, null_variance_status=var_status,
                    n_null_samples=k_null_samples, metadata={"n_tasks": n_tasks}
                )

                # --- Level 2 ---
                l1_valid = (empirical_p is not None and empirical_p <= STATISTICAL_P_THRESHOLD) or (computed_z is not None and abs(computed_z) >= 3.0)

                if l1_valid and abs(avg_raw) >= RAW_INFLUENCE_FLOOR:
                    z_deltas, out_deltas = [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        contrib_clean = compute_head_contribution(self.model, clean_cache, src[0], src[1])
                        contrib_corrupt = compute_head_contribution(self.model, corrupted_cache, src[0], src[1])
                        delta_contrib = contrib_clean - contrib_corrupt

                        def patch_res(act, hook):
                            act[0, -1, :] += delta_contrib
                            return act

                        target_z_patched = [None]
                        def cap_z(act, hook):
                            target_z_patched[0] = act[0, -1, tgt[1], :].clone()
                            return act

                        hooks = [(f"blocks.{tgt[0]}.hook_resid_pre", patch_res), (f"blocks.{tgt[0]}.attn.hook_z", cap_z)]
                        with torch.no_grad():
                            patched_logits = self.model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

                        correct_id = int(self.model.to_single_token(task["correct"]))
                        incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                        b_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
                        p_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

                        z_corrupt = corrupted_cache[f"blocks.{tgt[0]}.attn.hook_z"][0, -1, tgt[1], :]
                        z_deltas.append(float((target_z_patched[0] - z_corrupt).norm().item()))
                        out_deltas.append(p_diff - b_diff)

                    evidence_ladder.level2_residual_path = InterventionEvidence(
                        intervention_type="residual_pre_patch", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.hook_resid_pre",
                        target_delta=float(np.mean(z_deltas)), output_delta=float(np.mean(out_deltas)),
                        metadata={"n_tasks": n_tasks}
                    )

                    # --- Level 3 ---
                    nat_effs, clamp_effs, med_effs, ratios, directions = [], [], [], [], []
                    for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits, _ in task_data:
                        nat_e, clamp_e, med_e, ratio, dir_label = evaluate_attention_clamping_intervention(
                            self.model, src, tgt, task, clean_cache, corrupted_cache, base_logits, corrupted_toks
                        )
                        nat_effs.append(nat_e)
                        clamp_effs.append(clamp_e)
                        med_effs.append(med_e)
                        ratios.append(ratio)
                        directions.append(dir_label)

                    avg_med = float(np.mean(med_effs))
                    avg_nat = float(np.mean(nat_effs))
                    avg_ratio = (avg_med / abs(avg_nat)) if abs(avg_nat) > 1e-6 else 0.0

                    if abs(avg_med) < MEDIATION_EFFECT_FLOOR:
                        overall_dir = "NEUTRAL"
                    elif np.sign(avg_nat) == np.sign(avg_med):
                        overall_dir = "AMPLIFYING" if abs(avg_nat) > abs(float(np.mean(clamp_effs))) else "SUPPRESSIVE"
                    else:
                        overall_dir = "OPPOSING"

                    evidence_ladder.level3_attention_effect = InterventionEvidence(
                        intervention_type="interventional_attention_clamping", source=src, target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_pattern",
                        natural_effect=avg_nat,
                        clamped_effect=float(np.mean(clamp_effs)),
                        mediated_effect=avg_med,
                        mediation_ratio=avg_ratio,
                        direction_label=overall_dir,
                        metadata={"n_tasks": n_tasks}
                    )

                edge = EvidenceEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence_ladder)
                graph.add_edge(edge)

        return graph


# =============================================================================
# 3. Milestone 3: Predictive Self-Model Benchmark Engine
# =============================================================================

class SelfModelPredictor:
    """
    موتور پیش‌بینی خودشناس:
    پیش‌بینی اثر مداخلات ترکیبی ندیده صرفاً بر اساس گراف علّی ثبت‌شده (بدون اجرای مدل اصلی).
    """
    def __init__(self, circuit_graph: SelfCircuitGraph):
        self.graph = circuit_graph

    def predict_joint_intervention(self, source_nodes: List[Tuple[int, int]], target_node: Tuple[int, int]) -> float:
        predicted_effect = 0.0
        for src in source_nodes:
            edge = next((e for e in self.graph.edges if e.source == src and e.target == target_node), None)
            if edge and edge.evidence.level3_attention_effect:
                # برآورد اولیه: مجموع برداری اثرات طبیعی (Natural Effect) ثبت‌شده در گراف
                predicted_effect += edge.evidence.level3_attention_effect.natural_effect
        return predicted_effect


def run_predictive_benchmark(
    engine: EpistemicEvidenceEngine,
    predictor: SelfModelPredictor,
    joint_sources: List[Tuple[int, int]],
    target: Tuple[int, int],
    tasks: List[Dict]
) -> Dict[str, float]:
    """
    اجرای بنچمارک پیش‌بینی: مقایسه پیش‌بینی گراف علّی با مداخله هم‌زمان واقعی روی مدل.
    """
    l_tgt, h_tgt = target
    actual_effects = []

    # ۱. پیش‌بینی حاصل از Self-Model
    predicted_effect = predictor.predict_joint_intervention(joint_sources, target)

    # ۲. اجرای مداخله ترکیبی واقعی روی مدل اصلی (Ground Truth)
    for task in tasks:
        clean_toks = engine.model.to_tokens(task["clean"])
        corrupted_toks = engine.model.to_tokens(task["corrupted"])

        with torch.no_grad():
            _, clean_cache = engine.model.run_with_cache(clean_toks)
            base_logits, corrupted_cache = engine.model.run_with_cache(corrupted_toks)

        total_delta_contrib = 0.0
        for src in joint_sources:
            l_src, h_src = src
            contrib_clean = compute_head_contribution(engine.model, clean_cache, l_src, h_src)
            contrib_corrupt = compute_head_contribution(engine.model, corrupted_cache, l_src, h_src)
            total_delta_contrib += (contrib_clean - contrib_corrupt)

        def patch_joint_res_hook(act, hook):
            act[0, -1, :] += total_delta_contrib
            return act

        with torch.no_grad():
            patched_logits = engine.model.run_with_hooks(
                corrupted_toks,
                fwd_hooks=[(f"blocks.{l_tgt}.hook_resid_pre", patch_joint_res_hook)]
            )

        correct_id = int(engine.model.to_single_token(task["correct"]))
        incorrect_id = int(engine.model.to_single_token(task["incorrect"]))
        base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())
        patched_diff = float((patched_logits[0, -1, correct_id] - patched_logits[0, -1, incorrect_id]).item())

        actual_effects.append(patched_diff - base_diff)

    actual_mean = float(np.mean(actual_effects))
    mae = float(abs(actual_mean - predicted_effect))

    print("\n==================================================")
    print("🎯 MILESTONE 3: PREDICTIVE SELF-MODEL BENCHMARK RESULTS")
    print("==================================================")
    print(f"Joint Intervention Sources: {joint_sources} ──► Target: {target}")
    print(f"Actual Joint Effect (Ground Truth):   {actual_mean:+6.4f}")
    print(f"Self-Model Predicted Joint Effect:    {predicted_effect:+6.4f}")
    print(f"Mean Absolute Error (MAE):            {mae:6.4f}")
    print("==================================================\n")

    return {
        "actual_joint_effect": actual_mean,
        "predicted_joint_effect": predicted_effect,
        "mae": mae
    }


# =============================================================================
# 4. Main Execution
# =============================================================================

if __name__ == "__main__":
    engine = EpistemicEvidenceEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    # ۱. ساخت Evidence Graph
    evidence_graph = engine.execute_pipeline(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=DEFAULT_K_NULL_SAMPLES
    )

    evidence_graph.print_circuit_summary(n_tasks=len(IOI_TASKS))

    # ۲. اجرای بنچمارک پیش‌بینی Milestone 3
    predictor = SelfModelPredictor(evidence_graph)

    # سنجش پیش‌بینی برای مداخله هم‌زمان L9H9 و L10H7 روی L11H10
    joint_test_sources = [(9, 9), (10, 7)]
    target_node = (11, 10)

    run_predictive_benchmark(
        engine=engine,
        predictor=predictor,
        joint_sources=joint_test_sources,
        target=target_node,
        tasks=IOI_TASKS
    )

/tmp/ipykernel_837/3388424884.py:228: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [EPISTEMIC ENGINE] Running Pipeline for Regime 'R1_IOI' across 2 task(s)

🕸️ SELFCIRCUITGRAPH (EPISTEMIC EVIDENCE) - REGIME 'R1_IOI'
   Note: Evidence evaluated on 2 task(s). No broad generalization claimed.
Total Candidate Nodes Evaluated: 6
Total Tracked Edges: 14

  ★ Pair L7H9  ──► L8H6 
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: +0.0000 | Null Mean: -0.0000 (Std: 0.000016) | Z: +1.04 | Emp p: 0.077
     ├── Level 2 (Residual Path Influence):     N/A
     ├── Level 3 (Interventional Attn Effect):  N/A
     ├── Level 4 (Q/K/V Attribution):          [NOT TESTED]
     └── Level 5 (Mechanistic Edge):            [NOT ESTABLISHED]

  ★ Pair L7H9  ──► L8H10
     Status: [CANDIDATE]
     ├── Level 1 (Target Value Influence):     Raw: -0.0044 | Null Mean: +0.0003 (Std: 0.001922) | Z: -2.44 | Emp p: 0.031
     ├── Level 2 (Residual Path Influence):     N/A
     ├── Level 3 (Interventional Attn Effect): 